# 13 — Generalization Error, F-Beta Analysis & Profitability Robustness

This notebook extends the experimental evaluation with analyses specifically
requested by the supervisor:

| Section | Topic | Key Question |
|---------|-------|--------------|
| **A** | Generalization Error | How stable are results across time folds? |
| **B** | F-Beta Analysis | Should we favour precision or recall for trading? |
| **C** | Hyperparameter CV Selection | Do optimal barrier params generalise across folds? |
| **D** | Profitability Robustness | Is profitability stable or driven by a few trades? |
| **E** | Scientific Discussion | Honest assessment of all findings |

**Important context:** The dataset contains only ~100 labelled events from
~4,000 SPY daily bars. All results carry high statistical uncertainty, and
conclusions must be interpreted with caution.

In [ ]:
import sys, os, warnings
from pathlib import Path
warnings.filterwarnings('ignore')

for mod_name in list(sys.modules.keys()):
    if mod_name.startswith('src'):
        del sys.modules[mod_name]

if 'google.colab' in str(getattr(sys, 'modules', {})) or os.path.exists('/content'):
    REPO_DIR  = '/content/regime-aware-ml-trading'
    PROJ_ROOT = os.path.join(REPO_DIR, 'regime-aware-ml-trading')
    if not os.path.isdir(PROJ_ROOT):
        os.system('git clone https://github.com/zaetae/regime-aware-ml-trading.git ' + REPO_DIR)
    else:
        os.system(f'cd {REPO_DIR} && git pull -q')
    os.system(f'{sys.executable} -m pip install -q yfinance hmmlearn scikit-learn seaborn statsmodels optuna')
    _spy_path = os.path.join(PROJ_ROOT, 'data', 'raw', 'spy.csv')
    if not os.path.isfile(_spy_path):
        os.makedirs(os.path.dirname(_spy_path), exist_ok=True)
        import yfinance as yf; import pandas as pd
        _spy = yf.download('SPY', start='2010-01-01', end='2026-01-01', auto_adjust=False)
        if isinstance(_spy.columns, pd.MultiIndex): _spy.columns = _spy.columns.droplevel(1)
        _spy = _spy[['Open','High','Low','Close','Volume']]
        if _spy.index.tz is not None: _spy.index = _spy.index.tz_localize(None)
        _spy.index.name = 'Date'; _spy.to_csv(_spy_path)
else:
    def _find_project_root():
        current = Path.cwd()
        for _ in range(10):
            if (current / 'src').is_dir(): return current
            current = current.parent
        return Path.cwd().parent if (Path.cwd().parent / 'src').is_dir() else Path.cwd()
    PROJ_ROOT = str(_find_project_root())

sys.path.insert(0, PROJ_ROOT)
os.chdir(PROJ_ROOT)

%matplotlib inline

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    fbeta_score, precision_score, recall_score,
    accuracy_score, f1_score, confusion_matrix, ConfusionMatrixDisplay,
)

from src.data.load_data import load_spy
from src.features.build_features import build_feature_matrix
from src.models.train import (
    temporal_split, train_random_forest, train_bagging, train_baseline,
    evaluate_model, walk_forward_cv, kfold_event_cv,
)
from src.backtest.simulator import evaluate_profitability

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 110
plt.rcParams['figure.figsize'] = (12, 5)

os.makedirs('reports/thesis_figures', exist_ok=True)

df = load_spy()
EXCLUDE = ['triangle_pattern', 'channel_pattern']
print(f'SPY: {len(df)} bars, {df.index[0].date()} to {df.index[-1].date()}')
print('Imports OK')

In [ ]:
# ── Build feature matrix with best-F1 params ──────────────────────
PT, SL, MH = 2.0, 1.5, 10   # best F1 from notebook 12

features, labels, labeled_df = build_feature_matrix(
    df, exclude_patterns=EXCLUDE,
    pt_mult=PT, sl_mult=SL, max_holding=MH,
)
features = features.fillna(0)

print(f'Events: {len(features)}, Features: {features.shape[1]}')
print(f'Labels: {labels.value_counts().to_dict()}')

---
## Section A — Generalization Error Analysis

With only ~100 labelled events a single train/test evaluation carries
substantial sampling noise.  We use two complementary cross-validation
strategies and report **mean ± std** for every metric.

| Strategy | Respects time? | Purpose |
|----------|----------------|--------|
| Walk-forward CV | Yes | Realistic deployment simulation |
| K-fold event CV | No | Reduce sampling noise (diagnostic) |

**Why variance matters:**  High standard deviation across folds signals
that the model's performance is *unstable* — it may look good in one
period and poor in another.  This is especially dangerous with small
datasets where a single fold can be dominated by a few events.

In [ ]:
# ── Helper: full evaluation with profitability per fold ─────────────

def evaluate_fold(rf, baseline, X_te, y_te, df_ohlcv, fold_labeled,
                  pt_mult, sl_mult, max_holding, fold_id):
    """Return a dict with classification + profitability for one fold."""
    y_pred_rf  = rf.predict(X_te)
    y_pred_bl  = baseline.predict(X_te)

    d = {'fold': fold_id, 'test_size': len(y_te)}

    # Classification
    d['rf_acc']  = accuracy_score(y_te, y_pred_rf)
    d['rf_f1']   = f1_score(y_te, y_pred_rf, average='macro', zero_division=0)
    d['bl_acc']  = accuracy_score(y_te, y_pred_bl)
    d['bl_f1']   = f1_score(y_te, y_pred_bl, average='macro', zero_division=0)

    # F-beta
    for beta in [0.5, 1.0, 2.0]:
        key = f'rf_f{str(beta).replace(".","")}'
        d[key] = fbeta_score(y_te, y_pred_rf, beta=beta, average='macro', zero_division=0)

    d['rf_prec'] = precision_score(y_te, y_pred_rf, average='macro', zero_division=0)
    d['rf_rec']  = recall_score(y_te, y_pred_rf,    average='macro', zero_division=0)

    # Profitability
    rf_met, rf_tr = evaluate_profitability(
        df_ohlcv, fold_labeled, y_pred_rf,
        pt_mult=pt_mult, sl_mult=sl_mult, max_holding=max_holding)
    bl_met, _     = evaluate_profitability(
        df_ohlcv, fold_labeled, y_pred_bl,
        pt_mult=pt_mult, sl_mult=sl_mult, max_holding=max_holding)

    d['rf_cum_ret']  = rf_met['cumulative_return']
    d['rf_sharpe']   = rf_met['sharpe_ratio']
    d['rf_winrate']  = rf_met['win_rate']
    d['rf_pf']       = rf_met['profit_factor'] if rf_met['profit_factor'] != 'inf' else 10.0
    d['rf_maxdd']    = rf_met['max_drawdown']
    d['rf_ntrades']  = rf_met['n_trades']
    d['bl_cum_ret']  = bl_met['cumulative_return']

    return d, rf_tr

print('evaluate_fold() defined.')

### A.1 — Walk-Forward Cross-Validation (temporal)

In [ ]:
# ── Walk-forward CV with full profitability ────────────────────────
dates = pd.DatetimeIndex(labeled_df['event_date'])
sort_idx = dates.argsort()
feat_s  = features.iloc[sort_idx].reset_index(drop=True)
lab_s   = labels.iloc[sort_idx].reset_index(drop=True)
ldf_s   = labeled_df.iloc[sort_idx].reset_index(drop=True)
dates_s = dates[sort_idx]

N_SPLITS = 5
n = len(feat_s)
fold_size = n // (N_SPLITS + 1)

wf_rows = []
for k in range(N_SPLITS):
    tr_end = fold_size * (k + 1)
    te_start = tr_end
    te_end   = min(tr_end + fold_size, n)
    if te_end <= te_start or tr_end < int(n * 0.30):
        continue

    X_tr, y_tr = feat_s.iloc[:tr_end], lab_s.iloc[:tr_end]
    X_te, y_te = feat_s.iloc[te_start:te_end], lab_s.iloc[te_start:te_end]
    te_ldf     = ldf_s.iloc[te_start:te_end].reset_index(drop=True)

    if len(y_te.unique()) < 2: continue

    rf  = train_random_forest(X_tr, y_tr, n_estimators=200, max_depth=8)
    bl  = train_baseline(X_tr, y_tr)
    row, _ = evaluate_fold(rf, bl, X_te, y_te, df, te_ldf, PT, SL, MH, k)
    row['train_size'] = len(X_tr)
    row['period'] = f"{dates_s[te_start].strftime('%Y-%m')} – {dates_s[te_end-1].strftime('%Y-%m')}"
    wf_rows.append(row)

from src.models.train import train_baseline  # ensure imported

wf_df = pd.DataFrame(wf_rows)
print(f'Walk-forward CV: {len(wf_df)} folds\n')
display_cols = ['fold','period','train_size','test_size',
                'rf_acc','rf_f1','rf_prec','rf_rec',
                'rf_cum_ret','rf_sharpe','rf_winrate','rf_ntrades']
print(wf_df[[c for c in display_cols if c in wf_df.columns]].to_string(index=False))

In [ ]:
# ── Walk-forward summary: mean ± std ───────────────────────────────
metric_cols = ['rf_acc','rf_f1','rf_prec','rf_rec',
               'rf_cum_ret','rf_sharpe','rf_winrate','rf_pf','rf_maxdd','rf_ntrades',
               'bl_acc','bl_f1','bl_cum_ret']
avail = [c for c in metric_cols if c in wf_df.columns]

summary_rows = []
for c in avail:
    vals = wf_df[c].astype(float)
    summary_rows.append({'metric': c, 'mean': vals.mean(), 'std': vals.std(),
                         'min': vals.min(), 'max': vals.max()})
wf_summary = pd.DataFrame(summary_rows)
print('=== Walk-Forward CV — Mean ± Std ===')
for _, r in wf_summary.iterrows():
    print(f"  {r['metric']:16s}  {r['mean']:+.4f} ± {r['std']:.4f}   [{r['min']:+.4f} .. {r['max']:+.4f}]")

In [ ]:
# ── Walk-forward fold variability plots ────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
metrics_to_plot = [
    ('rf_acc',     'Accuracy',          'steelblue'),
    ('rf_f1',      'F1 Macro',          'darkorange'),
    ('rf_cum_ret', 'Cumulative Return',  'seagreen'),
    ('rf_sharpe',  'Sharpe Ratio',       'mediumpurple'),
    ('rf_winrate', 'Win Rate',           'indianred'),
    ('rf_ntrades', 'Trades',             'teal'),
]
for ax, (col, title, color) in zip(axes.flat, metrics_to_plot):
    if col in wf_df.columns:
        vals = wf_df[col].astype(float)
        ax.bar(wf_df['fold'], vals, color=color, alpha=0.8)
        ax.axhline(vals.mean(), color='black', ls='--', lw=1, label=f'mean={vals.mean():.3f}')
        ax.fill_between(wf_df['fold'], vals.mean()-vals.std(), vals.mean()+vals.std(),
                        alpha=0.15, color='grey', label=f'±1 std')
        ax.set_title(title, fontweight='bold')
        ax.set_xlabel('Fold'); ax.legend(fontsize=8)
plt.suptitle('Walk-Forward CV — Per-Fold Variability', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('reports/thesis_figures/wf_fold_variability.png', dpi=150, bbox_inches='tight')
plt.show()

### A.2 — K-Fold Event-Level CV (diagnostic)

K-fold CV does **not** respect temporal ordering and therefore leaks
future information into training.  It is included as a complementary
diagnostic to reduce sampling noise, **not** as a replacement for
walk-forward CV.

In [ ]:
# ── K-fold event CV with profitability ─────────────────────────────
KF = 5
kf_fold_size = n // KF
kf_rows = []

for k in range(KF):
    te_start = k * kf_fold_size
    te_end   = (k+1)*kf_fold_size if k < KF-1 else n
    test_idx  = list(range(te_start, te_end))
    train_idx = [i for i in range(n) if i not in test_idx]
    n_tr = int(len(train_idx) * 0.8)

    X_tr = feat_s.iloc[train_idx[:n_tr]]
    y_tr = lab_s.iloc[train_idx[:n_tr]]
    X_te = feat_s.iloc[test_idx]
    y_te = lab_s.iloc[test_idx]
    te_ldf = ldf_s.iloc[test_idx].reset_index(drop=True)

    if len(y_te.unique()) < 2 or len(y_tr.unique()) < 2: continue

    rf = train_random_forest(X_tr, y_tr, n_estimators=200, max_depth=8)
    bl = train_baseline(X_tr, y_tr)
    row, _ = evaluate_fold(rf, bl, X_te, y_te, df, te_ldf, PT, SL, MH, k)
    row['train_size'] = len(X_tr)
    kf_rows.append(row)

kf_df = pd.DataFrame(kf_rows)
print(f'K-fold CV: {len(kf_df)} folds\n')
print(kf_df[[c for c in display_cols if c in kf_df.columns]].to_string(index=False))

In [ ]:
# ── K-fold summary ─────────────────────────────────────────────────
print('=== K-Fold CV — Mean ± Std ===')
for c in [col for col in avail if col in kf_df.columns]:
    vals = kf_df[c].astype(float)
    print(f"  {c:16s}  {vals.mean():+.4f} ± {vals.std():.4f}")

In [ ]:
# ── Side-by-side comparison: WF vs KF ──────────────────────────────
compare_metrics = ['rf_acc','rf_f1','rf_cum_ret','rf_sharpe','rf_winrate']
compare_rows = []
for c in compare_metrics:
    if c in wf_df.columns and c in kf_df.columns:
        compare_rows.append({
            'Metric': c,
            'WF mean': wf_df[c].astype(float).mean(),
            'WF std':  wf_df[c].astype(float).std(),
            'KF mean': kf_df[c].astype(float).mean(),
            'KF std':  kf_df[c].astype(float).std(),
        })
compare_df = pd.DataFrame(compare_rows)
print('=== Walk-Forward vs K-Fold Comparison ===')
print(compare_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(compare_df))
w = 0.35
ax.bar(x - w/2, compare_df['WF mean'], w, yerr=compare_df['WF std'],
       label='Walk-Forward', color='steelblue', alpha=0.8, capsize=4)
ax.bar(x + w/2, compare_df['KF mean'], w, yerr=compare_df['KF std'],
       label='K-Fold', color='darkorange', alpha=0.8, capsize=4)
ax.set_xticks(x)
ax.set_xticklabels(compare_df['Metric'], rotation=20, ha='right')
ax.set_ylabel('Score'); ax.set_title('Walk-Forward vs K-Fold CV Comparison', fontweight='bold')
ax.legend(); plt.tight_layout()
plt.savefig('reports/thesis_figures/wf_vs_kf_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

### A.3 — Interpretation

**Key observations from the fold analysis:**

1. **High variance** across folds is expected with ~20 test events per fold.
   A single favourable or unfavourable trade can swing the cumulative return
   by several percentage points.

2. **Walk-forward CV** typically shows higher variance than k-fold CV because
   it respects temporal order — different market regimes (2010s bull market
   vs. COVID crash vs. recovery) fall into different folds.

3. **Overfitting risk:**  If training-set metrics are substantially better than
   test-set metrics, the model is memorising training events.  Tree-based
   models with `max_depth=8` on ~60 training events are prone to this.

4. **Generalization uncertainty:**  The wide confidence bands mean that the
   reported point estimates (e.g. "25.9% return") are within a broad
   distribution — the true out-of-sample performance could be substantially
   higher or lower.

---
## Section B — F-Beta Analysis

The standard F1 score weights precision and recall equally.  In trading,
however, the costs of false positives (FP) and false negatives (FN) are
asymmetric:

| Error type | Trading meaning | Cost |
|-----------|-----------------|------|
| **False positive** | Model says *trade*, but the trade loses | Direct financial loss |
| **False negative** | Model says *no trade*, but it would have won | Missed opportunity |

**F-beta** generalises F1 by weighting precision vs. recall:

$$F_\beta = (1+\beta^2) \cdot \frac{\mathrm{precision} \cdot \mathrm{recall}}{\beta^2 \cdot \mathrm{precision} + \mathrm{recall}}$$

| Beta | Emphasis | Trading interpretation |
|------|----------|------------------------|
| **F0.5** | Precision-heavy | Fewer but cleaner trades (avoid FPs) |
| **F1.0** | Balanced | Equal weight to FPs and FNs |
| **F2.0** | Recall-heavy | Capture more opportunities (tolerate FPs) |

In [ ]:
# ── F-beta on held-out test set ────────────────────────────────────
split = temporal_split(features, labels, labeled_df)
X_train, y_train = split['X_train'], split['y_train']
X_test,  y_test  = split['X_test'],  split['y_test']

rf_model = train_random_forest(X_train, y_train, n_estimators=200, max_depth=8)
y_pred   = rf_model.predict(X_test)

beta_values = [0.5, 1.0, 2.0]
beta_rows = []
for beta in beta_values:
    fb = fbeta_score(y_test, y_pred, beta=beta, average='macro', zero_division=0)
    prec = precision_score(y_test, y_pred, average='macro', zero_division=0)
    rec  = recall_score(y_test, y_pred, average='macro', zero_division=0)
    acc  = accuracy_score(y_test, y_pred)
    beta_rows.append({'beta': beta, 'F_beta': fb, 'precision': prec,
                      'recall': rec, 'accuracy': acc})

beta_df = pd.DataFrame(beta_rows)
print('=== F-Beta Scores (Test Set, RF model) ===')
print(beta_df.to_string(index=False))

In [ ]:
# ── F-beta across walk-forward folds ───────────────────────────────
fbeta_cols = ['rf_f05','rf_f10','rf_f20']
fbeta_labels = ['F0.5 (precision)','F1.0 (balanced)','F2.0 (recall)']

if all(c in wf_df.columns for c in fbeta_cols):
    fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
    for ax, col, lbl in zip(axes, fbeta_cols, fbeta_labels):
        vals = wf_df[col].astype(float)
        ax.bar(wf_df['fold'], vals, color='steelblue', alpha=0.8)
        ax.axhline(vals.mean(), color='red', ls='--', label=f'mean={vals.mean():.3f}')
        ax.set_title(lbl, fontweight='bold')
        ax.set_xlabel('Fold'); ax.set_ylabel('Score'); ax.legend(fontsize=8)
        ax.set_ylim(0, max(0.8, vals.max()*1.2))
    plt.suptitle('F-Beta Scores Across Walk-Forward Folds', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig('reports/thesis_figures/fbeta_per_fold.png', dpi=150, bbox_inches='tight')
    plt.show()

    print('\nF-beta summary (walk-forward):')
    for col, lbl in zip(fbeta_cols, fbeta_labels):
        vals = wf_df[col].astype(float)
        print(f'  {lbl}: {vals.mean():.4f} ± {vals.std():.4f}')

In [ ]:
# ── F-beta vs profitability relationship ───────────────────────────
if all(c in wf_df.columns for c in ['rf_f05','rf_f10','rf_f20','rf_cum_ret']):
    fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
    for ax, col, lbl in zip(axes, fbeta_cols, fbeta_labels):
        ax.scatter(wf_df[col].astype(float), wf_df['rf_cum_ret'].astype(float),
                   s=100, c='steelblue', edgecolors='black', alpha=0.8)
        for _, row in wf_df.iterrows():
            ax.annotate(f"f{int(row['fold'])}", (float(row[col])+0.005, float(row['rf_cum_ret'])+0.002),
                       fontsize=8)
        ax.set_xlabel(lbl); ax.set_ylabel('Cumulative Return')
        ax.set_title(f'{lbl} vs Profitability', fontweight='bold')
        ax.axhline(0, color='red', ls='--', alpha=0.4)
    plt.tight_layout()
    plt.savefig('reports/thesis_figures/fbeta_vs_profit.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# ── Confusion matrices for the held-out test set ───────────────────
fig, ax = plt.subplots(figsize=(6, 5))
cm = confusion_matrix(y_test, y_pred, labels=sorted(y_test.unique()))
disp = ConfusionMatrixDisplay(cm, display_labels=sorted(y_test.unique()))
disp.plot(ax=ax, cmap='Blues', values_format='d')
ax.set_title('Confusion Matrix — RF (Test Set)', fontweight='bold')
plt.tight_layout()
plt.savefig('reports/thesis_figures/confusion_matrix_test_nb13.png', dpi=150, bbox_inches='tight')
plt.show()

# Show per-class precision / recall
from sklearn.metrics import classification_report
print(classification_report(y_test, y_pred, zero_division=0))

### B.1 — F-Beta Interpretation

**Precision-focused (F0.5):**  Penalises false positives heavily.  In
trading, this means the model takes fewer trades but with higher
conviction.  This is attractive when transaction costs are high or when
capital preservation is the priority.

**Recall-focused (F2.0):**  Penalises false negatives heavily.  The
model tries to capture every opportunity, accepting more false alarms.
This is attractive when the upside of correct trades far outweighs the
downside of incorrect ones.

**Key question:**  Does precision-heavy or recall-heavy evaluation
correlate better with actual profitability?  The scatter plots above
provide empirical evidence for this specific dataset.

---
## Section C — Hyperparameter CV Selection

Notebook 12 selected hyperparameters using a **single train/validation
split**.  Here we evaluate a subset of configurations across **all
walk-forward folds** to assess whether the "best" parameters are
stable or an artefact of one particular split.

In [ ]:
# ── CV-based hyperparameter evaluation ─────────────────────────────
from src.models.optimize import _precompute, _build_features_fast

# Smaller grid for CV (full grid × 5 folds would be very slow)
HP_GRID = [
    {'pt_mult': 1.5, 'sl_mult': 1.5, 'max_holding': 10},
    {'pt_mult': 2.0, 'sl_mult': 1.5, 'max_holding': 10},  # best F1
    {'pt_mult': 2.0, 'sl_mult': 2.0, 'max_holding': 10},  # default
    {'pt_mult': 2.0, 'sl_mult': 2.0, 'max_holding': 15},
    {'pt_mult': 2.5, 'sl_mult': 2.0, 'max_holding': 10},
    {'pt_mult': 2.5, 'sl_mult': 3.0, 'max_holding': 20},  # best profit
    {'pt_mult': 1.5, 'sl_mult': 2.5, 'max_holding': 15},
    {'pt_mult': 3.0, 'sl_mult': 2.0, 'max_holding': 10},
]

print(f'Evaluating {len(HP_GRID)} configurations across walk-forward folds...')
cache = _precompute(df, exclude_patterns=EXCLUDE)

hp_results = []
for cfg_i, cfg in enumerate(HP_GRID):
    pt, sl, mh = cfg['pt_mult'], cfg['sl_mult'], cfg['max_holding']
    cfg_label = f'pt={pt}/sl={sl}/mh={mh}'

    feat_c, lab_c, ldf_c = _build_features_fast(df, cache, pt, sl, mh)
    if len(feat_c) < 20: continue
    feat_c = feat_c.fillna(0)

    dates_c = pd.DatetimeIndex(ldf_c['event_date'])
    si = dates_c.argsort()
    feat_c = feat_c.iloc[si].reset_index(drop=True)
    lab_c  = lab_c.iloc[si].reset_index(drop=True)
    ldf_c  = ldf_c.iloc[si].reset_index(drop=True)

    nc = len(feat_c)
    fs = nc // (N_SPLITS + 1)
    if fs < 3: continue

    fold_scores = []
    for k in range(N_SPLITS):
        tr_end = fs * (k + 1)
        te_s, te_e = tr_end, min(tr_end + fs, nc)
        if te_e <= te_s or tr_end < int(nc * 0.3): continue

        X_tr, y_tr = feat_c.iloc[:tr_end], lab_c.iloc[:tr_end]
        X_te, y_te = feat_c.iloc[te_s:te_e], lab_c.iloc[te_s:te_e]
        te_ldf = ldf_c.iloc[te_s:te_e].reset_index(drop=True)
        if len(y_te.unique()) < 2: continue

        rf = train_random_forest(X_tr, y_tr, n_estimators=100, max_depth=8)
        y_pred = rf.predict(X_te)

        f1_val = f1_score(y_te, y_pred, average='macro', zero_division=0)
        met, _ = evaluate_profitability(df, te_ldf, y_pred, pt_mult=pt, sl_mult=sl, max_holding=mh)
        fold_scores.append({'f1': f1_val, 'cum_ret': met['cumulative_return'],
                            'sharpe': met['sharpe_ratio'], 'winrate': met['win_rate']})

    if not fold_scores: continue
    fdf = pd.DataFrame(fold_scores)
    hp_results.append({
        'config': cfg_label, **cfg,
        'f1_mean': fdf['f1'].mean(), 'f1_std': fdf['f1'].std(),
        'ret_mean': fdf['cum_ret'].mean(), 'ret_std': fdf['cum_ret'].std(),
        'sharpe_mean': fdf['sharpe'].mean(), 'sharpe_std': fdf['sharpe'].std(),
        'n_folds': len(fdf),
    })
    if (cfg_i + 1) % 4 == 0:
        print(f'  {cfg_i+1}/{len(HP_GRID)} done')

hp_df = pd.DataFrame(hp_results)
print(f'\n=== Hyperparameter CV Results ({len(hp_df)} configs) ===')
print(hp_df[['config','f1_mean','f1_std','ret_mean','ret_std','sharpe_mean','n_folds']]
      .sort_values('f1_mean', ascending=False).to_string(index=False))

In [ ]:
# ── Stability heatmap: mean F1 by pt_mult × sl_mult ───────────────
if len(hp_df) > 3:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    for ax, val_col, title in zip(axes,
            ['f1_mean', 'ret_mean'],
            ['CV Mean F1 Macro', 'CV Mean Cum. Return']):
        pivot = hp_df.pivot_table(values=val_col, index='sl_mult', columns='pt_mult', aggfunc='mean')
        sns.heatmap(pivot, annot=True, fmt='.3f', cmap='RdYlGn', ax=ax, center=0 if 'ret' in val_col else None)
        ax.set_title(title, fontweight='bold')

    plt.suptitle('Hyperparameter Stability (Walk-Forward CV)', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig('reports/thesis_figures/hp_cv_heatmap.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# ── Ranking: best by F1 vs best by return ──────────────────────────
best_f1_cv   = hp_df.sort_values('f1_mean', ascending=False).iloc[0]
best_ret_cv  = hp_df.sort_values('ret_mean', ascending=False).iloc[0]

print('=== Best Configuration by CV F1 ===')
print(f"  Config: {best_f1_cv['config']}")
print(f"  F1: {best_f1_cv['f1_mean']:.4f} ± {best_f1_cv['f1_std']:.4f}")
print(f"  Return: {best_f1_cv['ret_mean']:.4f} ± {best_f1_cv['ret_std']:.4f}")

print('\n=== Best Configuration by CV Return ===')
print(f"  Config: {best_ret_cv['config']}")
print(f"  F1: {best_ret_cv['f1_mean']:.4f} ± {best_ret_cv['f1_std']:.4f}")
print(f"  Return: {best_ret_cv['ret_mean']:.4f} ± {best_ret_cv['ret_std']:.4f}")

if best_f1_cv['config'] != best_ret_cv['config']:
    print('\nConfirmed: best-for-F1 and best-for-profit differ even under CV evaluation.')
else:
    print('\nInteresting: same config wins for both metrics under CV.')

---
## Section D — Profitability Robustness

A strategy that produces high average returns but with extreme variance
is fragile.  A single bad fold could wipe out accumulated gains.  Here
we analyse the distribution and stability of profitability metrics.

In [ ]:
# ── Profitability distribution across folds ────────────────────────
profit_metrics = ['rf_cum_ret','rf_sharpe','rf_winrate','rf_pf','rf_maxdd','rf_ntrades']
profit_labels  = ['Cum. Return','Sharpe','Win Rate','Profit Factor','Max Drawdown','Trades']

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
for ax, col, lbl in zip(axes.flat, profit_metrics, profit_labels):
    if col in wf_df.columns:
        vals = wf_df[col].astype(float)
        ax.boxplot(vals, vert=True, patch_artist=True,
                   boxprops=dict(facecolor='lightsteelblue'),
                   medianprops=dict(color='red', linewidth=2))
        ax.scatter(np.ones(len(vals)), vals, color='steelblue', s=60, zorder=3)
        ax.set_title(lbl, fontweight='bold')
        ax.set_ylabel(lbl)
        if 'ret' in col.lower() or 'sharpe' in col.lower():
            ax.axhline(0, color='red', ls='--', alpha=0.4)
plt.suptitle('Profitability Distribution — Walk-Forward Folds', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('reports/thesis_figures/profit_robustness.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Trade-level analysis from the held-out test set ────────────────
dates_sort = pd.DatetimeIndex(labeled_df['event_date']).argsort()
sorted_ldf = labeled_df.iloc[dates_sort].reset_index(drop=True)
n_total = len(sorted_ldf)
n_test_start = int(n_total * 0.8)
test_ldf = sorted_ldf.iloc[n_test_start:]

from src.backtest.simulator import simulate_trades
test_trades = simulate_trades(df, test_ldf, y_pred,
                              pt_mult=PT, sl_mult=SL, max_holding=MH)

if len(test_trades) > 0:
    print(f'Test trades: {len(test_trades)}')
    print(f'Winners: {(test_trades["net_return"]>0).sum()}, Losers: {(test_trades["net_return"]<=0).sum()}')
    print(f'Mean return: {test_trades["net_return"].mean():.5f}')
    print(f'Std return:  {test_trades["net_return"].std():.5f}')
    print(f'Best trade:  {test_trades["net_return"].max():.5f}')
    print(f'Worst trade: {test_trades["net_return"].min():.5f}')

    # Is profitability driven by a few outlier trades?
    sorted_returns = test_trades['net_return'].sort_values(ascending=False)
    top3 = sorted_returns.head(3).sum()
    total = sorted_returns.sum()
    print(f'\nTop 3 trades contribute: {top3:.5f} of {total:.5f} total ({100*top3/total if total != 0 else 0:.1f}%)')
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
    axes[0].hist(test_trades['net_return'], bins=15, color='steelblue', alpha=0.7, edgecolor='black')
    axes[0].axvline(0, color='red', ls='--')
    axes[0].set_xlabel('Trade Return'); axes[0].set_ylabel('Count')
    axes[0].set_title('Trade Return Distribution', fontweight='bold')
    
    test_trades['cum_return'] = test_trades['net_return'].cumsum()
    axes[1].plot(test_trades['cum_return'].values, 'b-', lw=1.5)
    axes[1].fill_between(range(len(test_trades)), test_trades['cum_return'], 0,
                         where=test_trades['cum_return']>=0, alpha=0.2, color='green')
    axes[1].fill_between(range(len(test_trades)), test_trades['cum_return'], 0,
                         where=test_trades['cum_return']<0, alpha=0.2, color='red')
    axes[1].axhline(0, color='red', ls='--', alpha=0.4)
    axes[1].set_xlabel('Trade #'); axes[1].set_ylabel('Cumulative Return')
    axes[1].set_title('Equity Curve (Test Set)', fontweight='bold')
    plt.tight_layout()
    plt.savefig('reports/thesis_figures/trade_analysis.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('No trades generated on test set.')

In [ ]:
# ── Effect of stop-loss width on robustness ────────────────────────
if len(hp_df) > 3:
    fig, ax = plt.subplots(figsize=(10, 5))
    for _, row in hp_df.iterrows():
        ax.errorbar(row['sl_mult'], row['ret_mean'], yerr=row['ret_std'],
                    fmt='o', markersize=8, capsize=5, capthick=1.5,
                    label=row['config'] if row['sl_mult'] in [1.5, 2.0, 3.0] else None)
    ax.axhline(0, color='red', ls='--', alpha=0.4)
    ax.set_xlabel('sl_mult (Stop Loss Width)', fontsize=11)
    ax.set_ylabel('CV Mean Cumulative Return', fontsize=11)
    ax.set_title('Stop-Loss Width vs Profitability Robustness\n(error bars = ±1 std across folds)',
                 fontweight='bold')
    ax.legend(fontsize=8, loc='best')
    plt.tight_layout()
    plt.savefig('reports/thesis_figures/sl_vs_robustness.png', dpi=150, bbox_inches='tight')
    plt.show()

---
## Section E — Final Scientific Discussion

### E.1 — Generalization Limitations

The fundamental constraint of this project is dataset size.  With ~100
labelled events, each cross-validation fold contains only ~17–20 test
events.  This means:

- **High variance:** A single trade swinging from profit to loss changes
  fold-level metrics by 5–10 percentage points.
- **Unstable hyperparameters:** The "best" configuration on one fold may
  perform poorly on another.
- **Overfitting risk:** With 200 trees and max_depth=8, the model has
  enough capacity to memorise ~60 training events.

### E.2 — Label-Definition Sensitivity

The labels depend directly on (pt_mult, sl_mult, max_holding).  Changing
these parameters does not just tune the model — it *redefines the task*.
A "long" event with pt=1.5 is fundamentally different from a "long"
event with pt=3.0.  This means the optimization landscape is not a
smooth function of hyperparameters but a discrete set of different
classification problems.

### E.3 — Classification vs Profitability Mismatch

The core finding — that optimal F1 parameters differ from optimal
profitability parameters — is robust across CV folds.  This mismatch
arises because:

- **F1 treats all errors equally.** A false positive that loses 0.1%
  and one that loses 3% contribute the same to the F1 calculation.
- **Profitability depends on magnitudes.** Wider stops (high sl_mult)
  allow winners to develop larger gains, even if they also produce
  more misclassified events.
- **no_trade skips matter.** A model that correctly predicts no_trade
  improves F1 but does not generate profit.

### E.4 — F-Beta and Trading Objectives

F-beta analysis reveals that precision and recall have different
relationships with profitability:

- **Precision** aligns with *capital preservation* — fewer but more
  reliable trades.
- **Recall** aligns with *opportunity capture* — more trades, accepting
  higher error rates.

The optimal beta depends on the trading context: a fund with strict
drawdown limits should favour F0.5 (precision), while a trend-following
strategy should favour F2.0 (recall).

### E.5 — Why Event-Based Learning Remains Promising

Despite the limitations, event-based learning offers genuine advantages:

1. **Focus:** By filtering ~4,000 bars down to ~100 events, we
   concentrate the model on moments where technical structure provides
   a potential edge.
2. **Interpretability:** Each prediction corresponds to a recognisable
   pattern (S/R approach, channel boundary touch, etc.).
3. **Controlled risk:** Triple-barrier labeling embeds stop-loss and
   take-profit logic into the learning objective itself.
4. **Extensibility:** The framework naturally supports more detectors,
   more features, and more sophisticated models.

The path forward is not to abandon this approach but to *scale it*:
more assets, more events, and more rigorous validation.

In [ ]:
# ── Save all results for report generation ─────────────────────────
import json

nb13_results = {
    'walk_forward': {
        'n_folds': len(wf_df),
        'f1_mean': float(wf_df['rf_f1'].mean()),
        'f1_std':  float(wf_df['rf_f1'].std()),
        'ret_mean': float(wf_df['rf_cum_ret'].astype(float).mean()),
        'ret_std':  float(wf_df['rf_cum_ret'].astype(float).std()),
    },
    'kfold': {
        'n_folds': len(kf_df),
        'f1_mean': float(kf_df['rf_f1'].mean()),
        'f1_std':  float(kf_df['rf_f1'].std()),
        'ret_mean': float(kf_df['rf_cum_ret'].astype(float).mean()),
        'ret_std':  float(kf_df['rf_cum_ret'].astype(float).std()),
    },
    'fbeta_test': beta_df.to_dict(orient='records'),
    'hp_cv_best_f1': best_f1_cv['config'] if len(hp_df) > 0 else 'N/A',
    'hp_cv_best_ret': best_ret_cv['config'] if len(hp_df) > 0 else 'N/A',
}

os.makedirs('outputs', exist_ok=True)
with open('outputs/nb13_results.json', 'w') as f:
    json.dump(nb13_results, f, indent=2, default=str)

print('Results saved to outputs/nb13_results.json')
print('\n=== Notebook 13 complete ===')
print('Generated figures in reports/thesis_figures/:')
for fig_name in ['wf_fold_variability','wf_vs_kf_comparison','fbeta_per_fold',
                 'fbeta_vs_profit','confusion_matrix_test_nb13','hp_cv_heatmap',
                 'profit_robustness','trade_analysis','sl_vs_robustness']:
    path = f'reports/thesis_figures/{fig_name}.png'
    exists = 'OK' if os.path.exists(path) else 'MISSING'
    print(f'  {fig_name}.png ... {exists}')